# Assignment 1 — training pipeline

You're using a notebook for this stage. If you'd rather use a script,
write `train.py` instead and delete this file — `main.py` refuses to run
if it finds both `train.py` and `train.ipynb`, so exactly one of them must
exist.

Replace this cell and everything below it with your own code. When run via
`uv run python main.py train`, this notebook must, start to finish, with
no manual steps in between:

- Load your committed dataset.
- Apply the feature engineering, preprocessing, and train/val/test split
  consistent with the business framing you wrote in `REPORT.md`.
- Train the same model — linear regression for a regression target,
  logistic regression for a binary one — three ways, on the same split:
  scikit-learn, a from-scratch PyTorch loop (Session 5's manual approach:
  raw tensors, autograd, a manual gradient-step update), and the standard
  `torch.nn.Module` + `torch.optim` workflow.
- Evaluate all three against each other and against a naive baseline.
- Save whatever `app.py`/`app.ipynb` needs to build its dashboard without
  retraining anything.

How you structure the code beyond that — cell layout, function
boundaries, naming — is your call to make and be able to defend.

## Exploratory Data Analysis

In [1]:
import pandas as pd
import numpy as np

# Load our Bank Marketing dataset
df = pd.read_csv("data/bank-additional.csv", sep=";")

print("Dataset shape:", df.shape)
display(df.head())

print("\nColumn names and data types:")
print(df.dtypes)

Dataset shape: (4119, 21)


,age,job,marital,education,default,housing,loan,contact,month,day_of_week,...,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,30,blue-collar,married,basic.9y,no,yes,no,cellular,may,fri,...,2,999,0,nonexistent,-1.8,92.893,-46.2,1.313,5099.1,no
1,39,services,single,high.school,no,no,no,telephone,may,fri,...,4,999,0,nonexistent,1.1,93.994,-36.4,4.855,5191.0,no
2,25,services,married,high.school,no,yes,no,telephone,jun,wed,...,1,999,0,nonexistent,1.4,94.465,-41.8,4.962,5228.1,no
3,38,services,married,basic.9y,no,unknown,unknown,telephone,jun,fri,...,3,999,0,nonexistent,1.4,94.465,-41.8,4.959,5228.1,no
4,47,admin.,married,university.degree,no,yes,no,cellular,nov,mon,...,1,999,0,nonexistent,-0.1,93.200,-42.0,4.191,5195.8,no



Column names and data types:
age                 int64
job                   str
marital               str
education             str
default               str
housing               str
loan                  str
contact               str
month                 str
day_of_week           str
duration            int64
campaign            int64
pdays               int64
previous            int64
poutcome              str
emp.var.rate      float64
cons.price.idx    float64
cons.conf.idx     float64
euribor3m         float64
nr.employed       float64
y                     str
dtype: object


In [2]:
print("Target distribution:")
print(df["y"].value_counts())

print("\nTarget percentages:")
print(df["y"].value_counts(normalize=True) * 100)

Target distribution:
y
no     3668
yes     451
Name: count, dtype: int64

Target percentages:
y
no     89.05074
yes    10.94926
Name: proportion, dtype: float64


In [3]:
print("Missing values:")
print(df.isnull().sum())

print("\nUnknown values:")
print((df == "unknown").sum())

Missing values:
age               0
job               0
marital           0
education         0
default           0
housing           0
loan              0
contact           0
month             0
day_of_week       0
duration          0
campaign          0
pdays             0
previous          0
poutcome          0
emp.var.rate      0
cons.price.idx    0
cons.conf.idx     0
euribor3m         0
nr.employed       0
y                 0
dtype: int64

Unknown values:
age                 0
job                39
marital            11
education         167
default           803
housing           105
loan              105
contact             0
month               0
day_of_week         0
duration            0
campaign            0
pdays               0
previous            0
poutcome            0
emp.var.rate        0
cons.price.idx      0
cons.conf.idx       0
euribor3m           0
nr.employed         0
y                   0
dtype: int64


In [5]:
unknown_counts = (df == "unknown").sum()

print("Columns containing unknown values:")
print(unknown_counts[unknown_counts > 0].sort_values(ascending=False))

print("\nSpecial value in pdays:")
print(df["pdays"].value_counts().head(10))

Columns containing unknown values:
default      803
education    167
housing      105
loan         105
job           39
marital       11
dtype: int64

Special value in pdays:
pdays
999    3959
3        52
6        42
4        14
7        10
10        8
12        5
5         4
2         4
1         3
Name: count, dtype: int64


In [4]:
display(df.describe())

,age,duration,campaign,pdays,previous,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed
count,4119.000000,4119.000000,4119.000000,4119.000000,4119.000000,4119.000000,4119.000000,4119.000000,4119.000000,4119.000000
mean,40.113620,256.788055,2.537266,960.422190,0.190337,0.084972,93.579704,-40.499102,3.621356,5166.481695
std,10.313362,254.703736,2.568159,191.922786,0.541788,1.563114,0.579349,4.594578,1.733591,73.667904
min,18.000000,0.000000,1.000000,0.000000,0.000000,-3.400000,92.201000,-50.800000,0.635000,4963.600000
25%,32.000000,103.000000,1.000000,999.000000,0.000000,-1.800000,93.075000,-42.700000,1.334000,5099.100000
50%,38.000000,181.000000,2.000000,999.000000,0.000000,1.100000,93.749000,-41.800000,4.857000,5191.000000
75%,47.000000,317.000000,3.000000,999.000000,0.000000,1.400000,93.994000,-36.400000,4.961000,5228.100000
max,88.000000,3643.000000,35.000000,999.000000,6.000000,1.400000,94.767000,-26.900000,5.045000,5228.100000


### Exploratory Data Analysis Summary

- **Dataset:** 4,119 rows, 20 features, and 1 binary target (`y`).
- **Class imbalance:** 89.05% of customers did not subscribe, while only 10.95% did. Therefore, accuracy alone is not a reliable evaluation metric.
- **Missing values:** No `NaN` values, but some categorical features contain `"unknown"`, especially `default` (803 cases). We will initially keep these as separate categories.
- **Special values:** `pdays = 999` appears in 3,959 rows (96.1%) and indicates no previous campaign contact, rather than 999 actual days.
- **Data leakage:** We will remove `duration` because call length is unknown before contacting a customer.

## Feature Engineering

In [6]:
# Create a working copy of the original dataset
df_features = df.copy()

# Remove the target and call duration from the input features
X = df_features.drop(columns=["y", "duration"])

# Convert the binary target into numerical values
y = df_features["y"].map({"no": 0, "yes": 1})

print("Original shape:", df.shape)
print("Input feature shape:", X.shape)
print("Target shape:", y.shape)

Original shape: (4119, 21)
Input feature shape: (4119, 19)
Target shape: (4119,)


In [7]:
# Feature 1: Whether the customer was contacted previously
X["previously_contacted"] = (
    X["pdays"] != 999
).astype(int)

# Feature 2: Days since previous campaign contact
# Replace 999 with 0 because it means no prior contact
X["days_since_previous"] = X["pdays"].replace(999, 0)

# Feature 3: Whether the customer has any previous contact history
X["has_previous_contacts"] = (
    X["previous"] > 0
).astype(int)

# Remove original pdays to avoid treating 999 as an actual number
X = X.drop(columns=["pdays"])

display(
    X[
        [
            "previously_contacted",
            "days_since_previous",
            "previous",
            "has_previous_contacts"
        ]
    ].head(10)
)

,previously_contacted,days_since_previous,previous,has_previous_contacts
0,0,0,0,0
1,0,0,0,0
2,0,0,0,0
3,0,0,0,0
4,0,0,0,0
5,0,0,2,1
6,0,0,0,0
7,0,0,0,0
8,0,0,1,1
9,0,0,0,0


In [8]:
# Feature 4: Whether the customer has a housing or personal loan
X["has_any_loan"] = (
    (X["housing"] == "yes") |
    (X["loan"] == "yes")
).astype(int)

print("Customers by loan indicator:")
print(X["has_any_loan"].value_counts())

print("\nExample observations:")
display(
    X[
        ["housing", "loan", "has_any_loan"]
    ].head(10)
)

Customers by loan indicator:
has_any_loan
1    2430
0    1689
Name: count, dtype: int64

Example observations:


,housing,loan,has_any_loan
0,yes,no,1
1,no,no,0
2,yes,no,1
3,unknown,unknown,0
4,yes,no,1
5,no,no,0
6,yes,no,1
7,yes,no,1
8,no,no,0
9,no,no,0


In [9]:
print("Original input features:", df.shape[1] - 1)
print("Engineered input features:", X.shape[1])
print("Total rows:", X.shape[0])
print("Missing values:", X.isnull().sum().sum())

print("\nNew features:")
display(X[[
    "previously_contacted",
    "days_since_previous",
    "has_previous_contacts",
    "has_any_loan"
]].head())

Original input features: 20
Engineered input features: 22
Total rows: 4119
Missing values: 0

New features:


,previously_contacted,days_since_previous,has_previous_contacts,has_any_loan
0,0,0,0,1
1,0,0,0,0
2,0,0,0,1
3,0,0,0,0
4,0,0,0,1


### Feature Engineering Summary

I created four features based on previous campaign interactions and customers' financial characteristics:

- `previously_contacted`: Identifies customers with a recorded previous campaign contact based on `pdays`.
- `days_since_previous`: Preserves meaningful values from `pdays` without treating 999 as actual days.
- `has_previous_contacts`: Identifies customers who have been contacted in earlier campaigns.
- `has_any_loan`: Indicates whether a customer has at least one confirmed housing or personal loan.

I also removed `duration` to avoid data leakage and dropped the original `pdays` column after transforming it.

These features represent hypotheses that I will evaluate when comparing model performance.

## Train / Validation / Test Split
I split the dataset into 70% training, 15% validation, and 15% testing. I used stratified sampling because the target is imbalanced (approximately 89% no subscriptions and 11% subscriptions). This keeps a similar proportion of both classes across the three sets.

I also used random_state=42 to make the results reproducible.

In [10]:
from sklearn.model_selection import train_test_split

# First split: 70% training and 30% temporary
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

# Second split: divide the remaining 30% equally
# 15% validation and 15% testing
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

print("Training set:", X_train.shape)
print("Validation set:", X_val.shape)
print("Test set:", X_test.shape)

Training set: (2883, 22)
Validation set: (618, 22)
Test set: (618, 22)


In [11]:
print("TRAINING DISTRIBUTION:")
print(y_train.value_counts(normalize=True) * 100)

print("\nVALIDATION DISTRIBUTION:")
print(y_val.value_counts(normalize=True) * 100)

print("\nTEST DISTRIBUTION:")
print(y_test.value_counts(normalize=True) * 100)

TRAINING DISTRIBUTION:
y
0    89.039195
1    10.960805
Name: proportion, dtype: float64

VALIDATION DISTRIBUTION:
y
0    88.996764
1    11.003236
Name: proportion, dtype: float64

TEST DISTRIBUTION:
y
0    89.158576
1    10.841424
Name: proportion, dtype: float64


In [12]:
print(
    "Train and validation overlap:",
    len(set(X_train.index) & set(X_val.index))
)

print(
    "Train and test overlap:",
    len(set(X_train.index) & set(X_test.index))
)

print(
    "Validation and test overlap:",
    len(set(X_val.index) & set(X_test.index))
)

Train and validation overlap: 0
Train and test overlap: 0
Validation and test overlap: 0


### Split Summary

The dataset was divided into training (70%), validation (15%), and test (15%) sets using stratified sampling. All three sets maintain approximately the same subscription rate.

The training set will be used to fit the models, the validation set to compare configurations and select a decision threshold, and the test set for final evaluation.

A limitation is that this is a random row-level split rather than a chronological split, so it may not fully represent performance on future marketing campaigns.

## Data Preprocessing

In [13]:
# Identify column types
numerical_cols = X_train.select_dtypes(
    include=["int64", "float64", "int32", "float32"]
).columns.tolist()

categorical_cols = X_train.select_dtypes(
    include=["object", "str", "category"]
).columns.tolist()

print("Numerical features:", numerical_cols)
print("\nCategorical features:", categorical_cols)

print("\nNumerical count:", len(numerical_cols))
print("Categorical count:", len(categorical_cols))

Numerical features: ['age', 'campaign', 'previous', 'emp.var.rate', 'cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed', 'previously_contacted', 'days_since_previous', 'has_previous_contacts', 'has_any_loan']

Categorical features: ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'day_of_week', 'poutcome']

Numerical count: 12
Categorical count: 10


In [14]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore",
                              sparse_output=False), categorical_cols)
    ]
)

# Fit ONLY on training data
X_train_processed = preprocessor.fit_transform(X_train)

# Apply the same transformations to validation and test data
X_val_processed = preprocessor.transform(X_val)
X_test_processed = preprocessor.transform(X_test)

print("Training shape:", X_train_processed.shape)
print("Validation shape:", X_val_processed.shape)
print("Test shape:", X_test_processed.shape)

Training shape: (2883, 65)
Validation shape: (618, 65)
Test shape: (618, 65)


In [15]:
import numpy as np

print("All training values finite:",
      np.isfinite(X_train_processed).all())

print("All validation values finite:",
      np.isfinite(X_val_processed).all())

print("All test values finite:",
      np.isfinite(X_test_processed).all())

print("\nFinal feature count:", X_train_processed.shape[1])

All training values finite: True
All validation values finite: True
All test values finite: True

Final feature count: 65


### Preprocessing Summary

I identified 12 numerical and 10 categorical features. I used `StandardScaler` to standardize numerical variables and `OneHotEncoder` to convert categorical variables into numerical indicators.

After preprocessing, the dataset contains **65 features** instead of 22. The transformations were fitted only on the training set to avoid data leakage and then applied to the validation and test sets.

I also verified that all three processed datasets contain valid numerical values and are ready for model training.

## Model Training and Evaluation

### Naive Baseline

Because approximately 89% of customers did not subscribe, I will first evaluate a baseline model that always predicts the majority class (0, no subscription). This provides a reference for evaluating whether logistic regression is actually useful.

In [16]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

# Predict "no subscription" for every validation customer
baseline_predictions = np.zeros(len(y_val), dtype=int)

print("NAIVE BASELINE (VALIDATION)")
print("Accuracy:", accuracy_score(y_val, baseline_predictions))
print("Precision:", precision_score(y_val, baseline_predictions, zero_division=0))
print("Recall:", recall_score(y_val, baseline_predictions))
print("F1 Score:", f1_score(y_val, baseline_predictions))

NAIVE BASELINE (VALIDATION)
Accuracy: 0.889967637540453
Precision: 0.0
Recall: 0.0
F1 Score: 0.0


### Logistic Regression Model

In [17]:
from sklearn.linear_model import LogisticRegression

# Initialize the model
sklearn_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

# Train only on the training set
sklearn_model.fit(X_train_processed, y_train)

# Predict probabilities on validation data
sklearn_probabilities = sklearn_model.predict_proba(
    X_val_processed
)[:, 1]

# Classify using the initial threshold of 0.5
sklearn_predictions = (
    sklearn_probabilities >= 0.5
).astype(int)

print("Scikit-learn logistic regression trained successfully!")

Scikit-learn logistic regression trained successfully!


In [18]:
print("SCIKIT-LEARN LOGISTIC REGRESSION (VALIDATION)")

print("Accuracy:",
      accuracy_score(y_val, sklearn_predictions))

print("Precision:",
      precision_score(y_val, sklearn_predictions, zero_division=0))

print("Recall:",
      recall_score(y_val, sklearn_predictions))

print("F1 Score:",
      f1_score(y_val, sklearn_predictions))

print("ROC-AUC:",
      roc_auc_score(y_val, sklearn_probabilities))

SCIKIT-LEARN LOGISTIC REGRESSION (VALIDATION)
Accuracy: 0.8980582524271845
Precision: 0.6190476190476191
Recall: 0.19117647058823528
F1 Score: 0.29213483146067415
ROC-AUC: 0.7986631016042781


### Baseline vs. Scikit-learn Results

The naive baseline achieved 89% accuracy by always predicting no subscription, but had 0 precision, recall, and F1-score.

My logistic regression model achieved **89.81% accuracy, 61.90% precision, 19.12% recall, and 0.7987 ROC-AUC**.

The model is relatively precise when identifying potential subscribers, but its low recall means it misses most customers who actually subscribe. This suggests that the default probability threshold of 0.5 may not be ideal for the bank.

Later, I will compare different thresholds to find a better balance between unnecessary calls and missed opportunities.

### Manual PyTorch logistic regression

In [19]:
from sklearn.linear_model import LogisticRegression

sklearn_model = LogisticRegression(
    penalty=None,
    max_iter=2000,
    random_state=42
)

sklearn_model.fit(X_train_processed, y_train)

sklearn_probabilities = sklearn_model.predict_proba(
    X_val_processed
)[:, 1]

sklearn_predictions = (
    sklearn_probabilities >= 0.5
).astype(int)

/workspaces/daiist-assignment-1/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


In [20]:
import torch

torch.manual_seed(42)

# Use float64 for numerical consistency
X_train_tensor = torch.tensor(
    X_train_processed, dtype=torch.float64
)

y_train_tensor = torch.tensor(
    y_train.to_numpy(), dtype=torch.float64
)

X_val_tensor = torch.tensor(
    X_val_processed, dtype=torch.float64
)

print("Training tensor:", X_train_tensor.shape)
print("Target tensor:", y_train_tensor.shape)

Training tensor: torch.Size([2883, 65])
Target tensor: torch.Size([2883])


In [21]:
n_features = X_train_tensor.shape[1]

# Initialize parameters
weights = torch.zeros(
    n_features,
    dtype=torch.float64,
    requires_grad=True
)

bias = torch.zeros(
    1,
    dtype=torch.float64,
    requires_grad=True
)

print("Number of weights:", weights.numel())
print("Bias:", bias.item())

Number of weights: 65
Bias: 0.0


In [22]:
learning_rate = 0.1
epochs = 2000

loss_history = []

for epoch in range(epochs):

    # Step 1: Calculate raw prediction scores
    logits = X_train_tensor @ weights + bias

    # Step 2: Calculate the loss
    loss = torch.nn.functional.binary_cross_entropy_with_logits(
        logits, y_train_tensor
    )

    # Step 3: Calculate gradients
    loss.backward()

    # Step 4: Update parameters manually
    with torch.no_grad():
        weights -= learning_rate * weights.grad
        bias -= learning_rate * bias.grad

        # Clear gradients for the next iteration
        weights.grad.zero_()
        bias.grad.zero_()

    loss_history.append(loss.item())

    # Print progress every 400 epochs
    if (epoch + 1) % 400 == 0:
        print(f"Epoch {epoch + 1}: Loss = {loss.item():.4f}")

Epoch 400: Loss = 0.2733
Epoch 800: Loss = 0.2705
Epoch 1200: Loss = 0.2691
Epoch 1600: Loss = 0.2682
Epoch 2000: Loss = 0.2676


In [23]:
with torch.no_grad():
    val_logits = X_val_tensor @ weights + bias
    manual_probabilities = torch.sigmoid(val_logits).numpy()

manual_predictions = (
    manual_probabilities >= 0.5
).astype(int)

print("MANUAL PYTORCH (VALIDATION)")
print("Accuracy:", accuracy_score(y_val, manual_predictions))
print("Precision:", precision_score(y_val, manual_predictions, zero_division=0))
print("Recall:", recall_score(y_val, manual_predictions))
print("F1 Score:", f1_score(y_val, manual_predictions))
print("ROC-AUC:", roc_auc_score(y_val, manual_probabilities))

MANUAL PYTORCH (VALIDATION)
Accuracy: 0.9029126213592233
Precision: 0.75
Recall: 0.17647058823529413
F1 Score: 0.2857142857142857
ROC-AUC: 0.8093048128342245


### Manual PyTorch vs. Scikit-learn Logistic Regression

The manual PyTorch model achieved **90.29% accuracy, 75% precision, 17.65% recall, and 0.8093 ROC-AUC**.

Compared to my initial scikit-learn model (89.81% accuracy, 61.90% precision, 19.12% recall, and 0.7987 ROC-AUC), manual PyTorch achieved slightly higher accuracy, precision, and ROC-AUC, but lower recall.

Both models are relatively good at identifying customers who are likely to subscribe, but they miss most actual subscribers at the 0.5 threshold. The manual PyTorch model is more conservative, making fewer positive predictions but with greater precision.

Overall, both implementations show similar performance, suggesting that manual gradient descent is learning meaningful patterns. However, the comparison is preliminary because the models use different optimization approaches, and I still need to check whether they have converged.

### Standard PyTorch Logistic Regression

In [24]:
import torch.nn as nn

torch.manual_seed(42)

# Define the logistic regression model
class LogisticRegressionTorch(nn.Module):
    def __init__(self, input_features):
        super().__init__()

        # One linear layer with one output
        self.linear = nn.Linear(input_features, 1)

    def forward(self, x):
        return self.linear(x).squeeze(1)

# Initialize the model
torch_model = LogisticRegressionTorch(n_features).double()

print(torch_model)

LogisticRegressionTorch(
  (linear): Linear(in_features=65, out_features=1, bias=True)
)


In [25]:
# Binary classification loss
criterion = nn.BCEWithLogitsLoss()

# Stochastic Gradient Descent optimizer
optimizer = torch.optim.SGD(
    torch_model.parameters(),
    lr=0.1
)

# Same settings as manual PyTorch
epochs = 2000

standard_loss_history = []

print("Optimizer:", optimizer.__class__.__name__)
print("Learning rate:", optimizer.param_groups[0]["lr"])
print("Epochs:", epochs)

Optimizer: SGD
Learning rate: 0.1
Epochs: 2000


In [26]:
torch_model.train()

for epoch in range(epochs):

    # 1. Reset gradients from the previous iteration
    optimizer.zero_grad()

    # 2. Forward pass: calculate raw prediction scores
    logits = torch_model(X_train_tensor)

    # 3. Calculate the error
    loss = criterion(logits, y_train_tensor)

    # 4. Calculate gradients
    loss.backward()

    # 5. Update parameters automatically
    optimizer.step()

    # Save training loss
    standard_loss_history.append(loss.item())

    # Print progress every 400 epochs
    if (epoch + 1) % 400 == 0:
        print(f"Epoch {epoch + 1}: Loss = {loss.item():.4f}")

Epoch 400: Loss = 0.2748
Epoch 800: Loss = 0.2715
Epoch 1200: Loss = 0.2699
Epoch 1600: Loss = 0.2688
Epoch 2000: Loss = 0.2681


In [27]:
# Switch to evaluation mode
torch_model.eval()

with torch.no_grad():

    # Calculate model logits
    standard_logits = torch_model(X_val_tensor)

    # Convert logits into probabilities
    standard_probabilities = torch.sigmoid(
        standard_logits
    ).numpy()

# Convert probabilities into predictions
standard_predictions = (
    standard_probabilities >= 0.5
).astype(int)

print("STANDARD PYTORCH (VALIDATION)")

print("Accuracy:",
      accuracy_score(y_val, standard_predictions))

print("Precision:",
      precision_score(
          y_val, standard_predictions, zero_division=0
      ))

print("Recall:",
      recall_score(y_val, standard_predictions))

print("F1 Score:",
      f1_score(y_val, standard_predictions))

print("ROC-AUC:",
      roc_auc_score(y_val, standard_probabilities))

STANDARD PYTORCH (VALIDATION)
Accuracy: 0.9029126213592233
Precision: 0.75
Recall: 0.17647058823529413
F1 Score: 0.2857142857142857
ROC-AUC: 0.8087700534759358


### Standard PyTorch Implementation — Summary

This implementation uses `nn.Module` and `nn.Linear` to define the logistic regression model, and `torch.optim.SGD` to update its parameters.

Unlike the manual PyTorch approach, the optimizer handles the gradient updates automatically. I used the same preprocessing, training data, learning rate, number of epochs, and loss function to make the comparison as fair as possible.

I will compare all three models on the validation set to determine whether their performance is similar and investigate any differences.

## Comparison of all three implementations

In [30]:
import pandas as pd

def evaluate_model(y_true, predictions, probabilities):
    return {
        "Accuracy": accuracy_score(y_true, predictions),
        "Precision": precision_score(
            y_true, predictions, zero_division=0
        ),
        "Recall": recall_score(y_true, predictions),
        "F1": f1_score(y_true, predictions),
        "ROC-AUC": roc_auc_score(y_true, probabilities)
    }

results = pd.DataFrame({
    "Naive Baseline": evaluate_model(
        y_val, baseline_predictions, baseline_predictions
    ),
    "Scikit-learn": evaluate_model(
        y_val, sklearn_predictions, sklearn_probabilities
    ),
    "Manual PyTorch": evaluate_model(
        y_val, manual_predictions, manual_probabilities
    ),
    "Standard PyTorch": evaluate_model(
        y_val, standard_predictions, standard_probabilities
    )
}).T

display(results.round(4))

,Accuracy,Precision,Recall,F1,ROC-AUC
Naive Baseline,0.8900,0.0000,0.0000,0.0000,0.5000
Scikit-learn,0.9029,0.6538,0.2500,0.3617,0.7916
Manual PyTorch,0.9029,0.7500,0.1765,0.2857,0.8093
Standard PyTorch,0.9029,0.7500,0.1765,0.2857,0.8088


### Model Comparison

The naive baseline achieved **89% accuracy**, but 0 precision and recall, showing why accuracy alone is misleading for imbalanced data.

All three logistic regression models improved on the baseline, achieving **90.29% accuracy**. Scikit-learn had the highest recall (25%) and F1-score (0.3617), while both PyTorch models had higher precision (75%) and ROC-AUC (around 0.81).

The two PyTorch implementations produced nearly identical results, suggesting that the manual implementation works correctly.

Overall, all three models outperform the baseline, but their low recall suggests that **adjusting the classification threshold** could help identify more potential subscribers.